## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 7 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Tim data engineering platform e-commerce meminta anda membangun **pipeline ETL produksi pertama** yang menggabungkan tiga sumber data sekaligus: transaksi (CSV), data produk (JSON), dan data ulasan pelanggan (CSV terpisah) — merepresentasikan kondisi nyata di mana data tersebar di berbagai sistem dengan format berbeda-beda.

### Menyiapkan Dataset

Jalankan cell berikut untuk menghasilkan **tiga sumber data** sekaligus.

In [2]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


### Instruksi Pengerjaan

Buat notebook baru **`Tugas6_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession`, lalu bangun pipeline ETL lengkap mengikuti struktur **Extract → Transform → Load**:

---

**A. EXTRACT** *(bobot 15%)*

Baca ketiga sumber data (`tugas6_transaksi.csv`, `tugas6_produk.json`, `tugas6_ulasan.csv`) menjadi tiga Spark DataFrame terpisah. Tampilkan jumlah baris dan `printSchema()` masing-masing.

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

# Inisialisasi Spark Session
spark = SparkSession.builder.appName("TugasMandiriETL").getOrCreate()

# A. EXTRACT
df_trx = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk = spark.read.json("tugas6_produk.json")
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

print(f"Jumlah transaksi : {df_trx.count()}")
print(f"Jumlah produk    : {df_produk.count()}")
print(f"Jumlah ulasan    : {df_ulasan.count()}")

26/10/01 14:38:40 WARN Utils: Your hostname, aramsamsam-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/10/01 14:38:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/01 14:38:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/01 14:38:46 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
                                                                                

Jumlah transaksi : 5000
Jumlah produk    : 30
Jumlah ulasan    : 3500


**B. TRANSFORM — Penggabungan** *(bobot 25%)*

Gabungkan ketiga DataFrame menjadi satu (`order_id` sebagai kunci ke ulasan, `product_id` sebagai kunci ke produk). Gunakan **`salah satu join yang tepat`** dari transaksi ke ulasan (karena tidak semua transaksi memiliki ulasan) dan **`salah satu join yang tepat`** dari transaksi ke produk (karena setiap transaksi pasti memiliki produk yang valid). Tambahkan kolom `total_pendapatan` (`unit_terjual x harga`).

**C. TRANSFORM — Penanganan Data Kosong & Pengayaan** *(bobot 20%)*

- Transaksi tanpa ulasan akan memiliki `rating` bernilai kosong (`null`) setelah `salah satu join yang tepat` — isi nilai kosong tersebut dengan angka **0** menggunakan `salah satu function`, sertakan alasan singkat mengapa 0 (bukan nilai lain) masuk akal untuk kasus "belum ada ulasan".
- Tambahkan kolom `ada_ulasan` bernilai `True`/`False` (tidak boleh diisi manual satu satu)`, **sebelum** langkah `na.fill()` di atas).

In [8]:
# B & C. TRANSFORM (Join, Kolom Turunan, Handling Null)
# Step 1: Join transaksi ke produk (Inner Join)
df_transformed = df_trx.join(df_produk, on="product_id", how="inner")

# Step 2: Join transaksi ke ulasan (Left Join)
df_transformed = df_transformed.join(df_ulasan, on="order_id", how="left")

# Step 3: Hitung total pendapatan
df_transformed = df_transformed.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))

# Step 4: Tambahkan flag 'ada_ulasan' SEBELUM fillna
df_transformed = df_transformed.withColumn("ada_ulasan", col("rating").isNotNull())

# Step 5: Handling Missing Value rating dengan angka 0
# Alasan: Angka 0 merepresentasikan kondisi faktual bahwa belum ada ulasan/rating dari pembeli.
df_transformed = df_transformed.fillna({"rating": 0})

**D. LOAD** *(bobot 25%)*

Simpan hasil akhir ke HDFS dalam format **Parquet**, dipartisi berdasarkan `kategori`, ke path `/user/[username]/tugas6/hasil_etl`. Verifikasi dengan `hdfs dfs -ls -R`, lalu baca kembali dan tampilkan `count()`-nya sebagai bukti data tersimpan utuh.


In [13]:
# D. LOAD KE HDFS
hdfs_path = "hdfs://localhost:9000/user/aramsamsam/tugas6/hasil_etl"
df_transformed.write.mode("overwrite").partitionBy("kategori").parquet(hdfs_path)

# Verifikasi Load
df_verify = spark.read.parquet(hdfs_path)
print("Jumlah total baris tersimpan di HDFS:", df_verify.count())

Jumlah total baris tersimpan di HDFS: 5000


**E. Insight Akhir** *(bobot 15%)*

Dari data hasil ETL, tampilkan (menggunakan DataFrame API **atau** Spark SQL, bebas memilih): kategori produk mana yang memiliki **persentase transaksi dengan ulasan** (`ada_ulasan = True`) **paling rendah**? Tulis 2-3 kalimat interpretasi bisnis pada markdown cell: mengapa hal ini mungkin penting diketahui oleh tim marketing?


In [14]:
from pyspark.sql.functions import avg

# Analisis persentase ulasan per kategori
df_verify.groupBy("kategori").agg(
    (avg(col("ada_ulasan").cast("integer")) * 100).alias("persentase_ulasan")
).orderBy("persentase_ulasan").show()

[Stage 35:>                                                         (0 + 3) / 3]

+------------+-----------------+
|    kategori|persentase_ulasan|
+------------+-----------------+
|     Makanan|68.84328358208955|
|   Kesehatan|68.88657648283039|
|     Fashion|70.14492753623188|
|Rumah Tangga| 70.5521472392638|
|  Elektronik|  70.659407138536|
+------------+-----------------+

